In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


# Measurement-Channel FDIA Localization / 量测通道定位评估

This notebook evaluates localization in measurement space. Detection evaluations are in `experiments/02_detection/`. All methods below receive the same pre-generated attacked windows, and only the attacked last frame is scored. / 本 Notebook 评估量测通道定位；检测入口位于检测目录，各方法使用相同的攻击窗口，仅对最后时刻评分。

In [ ]:
# Cell 1 - environment and paths
import os
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
os.environ.setdefault("PGAE_CASE", "case30")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/detection5-mpl")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/detection5-cache")
if not hasattr(np, "in1d"):
    np.in1d = np.isin
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULT_DIR = PROJECT_ROOT / "outputs" / "detection5" / os.environ["PGAE_CASE"]
RESULT_JSON = RESULT_DIR / "localization_results.json"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project: {PROJECT_ROOT}")

In [ ]:
# Cell 2 - explicit model imports used by the benchmark
from models.baselines.convae import Conv_AE
from models.baselines.lstmae import Encoder as LSTMEncoder
from models.baselines.anomaly_transformer import AnomTrans
from models.baselines.piconvae import PIConvAE
from models.baselines.revgae import ReVGAE
from models.baselines.at_gvae import ATGVAE
from models.detection.pgae_tau import PGAE_tau

In [ ]:
# Cell 3 - model/checkpoint mapping (kept separate for auditability)
MODEL_REGISTRY = {
    "ConvAE": {"class": "models.baselines.convae.Conv_AE", "checkpoint": "saved_model/{case}/ConvAE.pt", "score_type": "measurement_reconstruction"},
    "LSTMAE": {"class": "models.baselines.lstmae.LSTM_AE (checkpoint-compatible adapter)", "checkpoint": "saved_model/{case}/LSTMAE.pt", "score_type": "measurement_reconstruction"},
    "AT": {"class": "models.baselines.anomaly_transformer.AnomTrans", "checkpoint": "saved_model/{case}/AnomTrans.pt", "score_type": "measurement_reconstruction"},
    "PIConvAE": {"class": "models.baselines.piconvae.PIConvAE", "checkpoint": "saved_model/{case}/PIConvAE.pt", "score_type": "measurement_reconstruction"},
    "ReVGAE": {"class": "models.baselines.revgae.ReVGAE", "checkpoint": "saved_model/{case}/ReVGAE.pt", "score_type": "measurement_node_reconstruction"},
    "AT-GVAE": {"class": "models.baselines.at_gvae.ATGVAE", "checkpoint": "saved_model/{case}/AT_GVAE.pt", "score_type": "measurement_node_reconstruction"},
    "PGAE-tau": {"class": "models.detection.pgae_tau.PGAE_tau", "checkpoint": "saved_model/{case}/PGAE_tau.pt", "score_type": "physics_residual"},
}
MODEL_REGISTRY

In [ ]:
# Experiment configuration for measurement-channel localization
ANG_NO_LIST = [1, 2, 3]
MAG_NO = 0
ANG_STR_LIST = [0.2, 0.3]
MAG_STR = 0
STRENGTH_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
SINGLE_SAMPLE = {"ang_no": 3, "ang_str": 0.30, "mag_no": 0, "mag_str": 0}
N_SAMPLES = 200
SEED = 42
K_VALUES = [5, 10, 15, 20, 30]

EXPERIMENT = {
    "main": {"ang_no_list": ANG_NO_LIST, "ang_str_list": ANG_STR_LIST, "mag_no": MAG_NO, "mag_str": MAG_STR},
    "strength_sweep": {"ang_no_list": ANG_NO_LIST, "ang_str_list": STRENGTH_VALUES},
    "single_sample": SINGLE_SAMPLE,
}
EXPERIMENT

In [ ]:
# Cell 5 - execute the shared measurement-level localization benchmark
from evaluation.localization import run_localization_benchmark

results = run_localization_benchmark(
    project_root=str(PROJECT_ROOT),
    output_json=str(RESULT_JSON),
    model_registry=MODEL_REGISTRY,
    n_samples=N_SAMPLES,
    seed=SEED,
    batch_size=32,
    k_values=K_VALUES,
    main_ang_no=ANG_NO_LIST,
    main_ang_str=ANG_STR_LIST,
    strength_values=STRENGTH_VALUES,
)

In [ ]:
# Results are stored as portable JSON.
print(f"Saved: {RESULT_JSON}")


In [ ]:
# Cell 7 - compact execution summary
for row in results["model_status"]:
    suffix = f" - {row['reason']}" if row["reason"] else ""
    print(f"{row['model']}: {row['status']}{suffix}")
print(f"Metric rows: {len(results['metrics'])}")
print(f"Single-sample measurement rows: {len(results['single_sample'])}")